# 01: 動画 → 一次元輝度分布の抽出（低解像度・3チャンバー同時撮影系）

対流観察動画（MP4）から、各チャンバーの一次元輝度分布 L(x, t) を抽出してCSVに保存するノートブック。

**処理の流れ**
1. ffmpeg で動画から5秒間隔のフレームを抽出（§1）
2. 90°回転＋グレースケール化（サンプルによっては微小回転補正・画像レベル背景除去）（§2）
3. キャリブレーション線・抽出線の目視確認（§3）
4. 底面から柱高さの1/2上（水深10 mmに対して7.5 mm相当）の水平線上で輝度を抽出し、チャンバーごとにCSV保存（§4）

**使い方**
1. 「パス設定」セルで `RAW_VIDEO_ROOT` を自分の環境に合わせて書き換える
2. `SAMPLE_ID` を `../config/samples_lowres.json` のキーから1つ選ぶ
3. Run All

**前提**: ffmpeg がPATHに通っていること（READMEのセットアップ手順を参照）。
サンプル別のキャリブレーション座標・条件・旧ファイル名との対応は、すべて `config/samples_lowres.json` に記録されている。

## セットアップ

In [ ]:
import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from PIL import Image, ImageOps
from IPython.display import display

In [ ]:
# ============ パス設定（環境に合わせて書き換える） ============

# 生動画の置き場所。{date}/ 以下に動画ファイルがある構成を想定
#   例: RAW_VIDEO_ROOT/251117/149A3606.MP4
RAW_VIDEO_ROOT = Path("D:/kyoto_univ/raw_videos/251117")          # ★要変更

FRAMES_ROOT = Path("D:/kyoto_univ/tempdata/251117/frames_lowres")      # 抽出フレームの一時置き場（サイズ大・git管理外）
OUTPUT_ROOT = Path("../data/luminance_1d")       # 一次元輝度分布CSVの出力先
CONFIG_PATH = Path("../config/samples_lowres.json")

# ============ サンプル選択 ============
SAMPLE_ID = "251117_149A3606"                    # ★処理したいサンプルをconfigのキーから選ぶ

with open(CONFIG_PATH, encoding="utf-8") as f:
    config = json.load(f)

print("利用可能なサンプル:")
for k, v in config["samples"].items():
    mark = " (TODO: 座標未転記)" if v["chambers"] is None else ""
    print(f"  {k}{mark}")

cfg = config["samples"][SAMPLE_ID]
defaults = config["defaults"]
assert cfg["chambers"] is not None, f"{SAMPLE_ID} は座標未転記のプレースホルダです"

print(f"\n選択中: {SAMPLE_ID}")
print(json.dumps({c: v["condition"] for c, v in cfg["chambers"].items()},
                 ensure_ascii=False, indent=2))

## 関数群

※ ノートブック完結型の方針のため、共通関数は 01 / 02 の各ノートブック冒頭に同一のコピーを持つ。
**修正する場合は必ず両方を同時に修正すること**（README参照）。

In [ ]:
def parse_sec(path: Path) -> int:
    """フレームファイル名 'frame_{sec}sec_*.png' から秒数を取り出す。"""
    name = path.name
    return int(name.split("_")[1].replace("sec", ""))


def extract_frames_ffmpeg(video_path: Path, frames_dir: Path,
                          fps: int, time_period: int, start_sec: int, end_sec: int) -> None:
    """動画から time_period 秒間隔で1フレームずつPNGを抽出する。

    旧コードは各時刻で2フレーム(-vframes 2)を抽出し、後段で1枚おきに間引いて
    各時刻の1枚目のみを使っていた。ここでは等価な処理として最初から1枚のみ抽出する。
    """
    frames_dir.mkdir(parents=True, exist_ok=True)
    for start_time in range(start_sec, end_sec, time_period):
        output_path = frames_dir / f"frame_{start_time}sec_%2d.png"
        cmd = [
            "ffmpeg",
            "-ss", str(start_time),   # 先にシークして高速化
            "-i", str(video_path),
            "-vf", f"fps={fps}",
            "-vframes", "1",
            "-vsync", "vfr",          # 不要なフレームの生成を防ぐ
            str(output_path),
        ]
        subprocess.run(cmd, capture_output=True, text=True)


def load_grayscale_frames(frames_dir: Path, base_rotation_deg: int = 90,
                          fine_rotation_deg: float = 0.0) -> list:
    """フレームPNGを時系列順に読み込み、回転＋グレースケール化して返す。

    旧コードは os.listdir の返却順（環境依存）に頼っていたが、
    ここではファイル名の秒数で明示的に数値ソートし、時系列順を保証する。
    """
    paths = sorted(frames_dir.glob("*.png"), key=parse_sec)
    images = []
    for p in paths:
        img = Image.open(p)
        img = img.rotate(base_rotation_deg, expand=True)
        if fine_rotation_deg:
            img = img.rotate(fine_rotation_deg, expand=True)  # 例: -1 は旧コードの rotate(359) と等価
        images.append(ImageOps.grayscale(img))
    return images


def remove_background_min(images: list) -> list:
    """画像レベル背景除去（0320ロット互換）: 各画素の時系列最小値を背景として減算する。"""
    stack = np.stack([np.array(img) for img in images]).astype(np.int16)
    bg = stack.min(axis=0)
    removed = np.clip(stack - bg, 0, 255).astype(np.uint8)
    return [Image.fromarray(a) for a in removed]


def show_line_overlay(img, line_position, title=""):
    """画像に線分 [x1, y1, x2, y2] を重ねて表示する（キャリブレーション確認用）。"""
    plt.imshow(img, cmap="gray")
    plt.grid(True)
    plt.axis("on")
    line = mlines.Line2D([line_position[0], line_position[2]],
                         [line_position[1], line_position[3]],
                         linewidth=2, color="r")
    plt.gca().add_line(line)
    if title:
        plt.title(title)


def extract_luminance_from_line(grayscale_images, line_segment):
    """画像リストから指定した水平線分 (x1, y1, x2, y2) 上の輝度値を抽出する。

    Parameters
    ----------
    grayscale_images : list of PIL.Image
        グレースケール画像の時系列リスト
    line_segment : list or tuple
        (x1, y1, x2, y2)

    Returns
    -------
    list of list
        各時刻の一次元輝度分布 L(x)
    """
    extracted_luminance = []
    for img in grayscale_images:
        img_array = np.array(img)
        x1, y1, x2, y2 = (round(v) for v in line_segment)
        extracted_luminance.append(img_array[y1, x1:x2 + 1].tolist())
    return extracted_luminance

## §1 フレーム抽出

動画から5秒間隔でフレームを抽出する。既に抽出済み（フォルダが存在する）の場合はスキップする。

In [ ]:
video_path = RAW_VIDEO_ROOT / cfg["date"] / cfg["video_file"]
frames_dir = FRAMES_ROOT / SAMPLE_ID

if frames_dir.exists():
    print(f"Skip: フレームフォルダが既に存在します -> {frames_dir}")
else:
    assert video_path.exists(), f"動画が見つかりません: {video_path}"
    extract_frames_ffmpeg(
        video_path, frames_dir,
        fps=defaults["fps"], time_period=defaults["time_period_sec"],
        start_sec=defaults["start_sec"], end_sec=defaults["end_sec"],
    )
    print(f"抽出完了 -> {frames_dir}")

print(f"フレーム数: {len(list(frames_dir.glob('*.png')))}")

## §2 前処理（回転・グレースケール化）

90°回転＋グレースケール化を行う。configで指定がある場合のみ、微小回転補正（例: 251117_149A3608 の -1°）
および画像レベル背景除去（0320ロットのみ）を適用する。

In [ ]:
images = load_grayscale_frames(
    frames_dir,
    base_rotation_deg=defaults["base_rotation_deg"],
    fine_rotation_deg=cfg["fine_rotation_deg"],
)
print(f"読み込み画像数: {len(images)}（5秒間隔・時系列順）")

if cfg["image_level_bg_removal"]:
    print("画像レベル背景除去を適用します（0320ロット互換）")
    images = remove_background_min(images)

display(images[-1])

## §3 キャリブレーション確認

上段: 柱高さ（基準5 mm）に合わせた較正線。px/mm換算の根拠となる。
下段: 一次元輝度分布 L(x) の抽出線（較正線上端から柱高さの1/2上 ＝ 底面から7.5 mm相当）。
いずれの座標も `config/samples_lowres.json` に記録された手動指定値であり、赤線が意図した位置に
重なっていることを目視で確認する。

In [ ]:
CHAMBERS = ["top", "middle", "bottom"]

plt.figure(figsize=(20, 15))
for ch in CHAMBERS:
    cal = cfg["chambers"][ch]["calibration_line_px"]
    show_line_overlay(images[-1], cal, title=f"calibration: {ch}")
    h_px = cal[3] - cal[1]
    print(f"[{ch}] 較正線長 = {h_px} px "
          f"(-> {h_px / defaults['pillar_height_mm_for_calibration']:.1f} px/mm 基準)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(20, 15))
for ch in CHAMBERS:
    ext = cfg["chambers"][ch]["extraction_line_px"]
    show_line_overlay(images[-1], ext, title=f"extraction line: {ch}")
    print(f"[{ch}] 抽出線: y = {ext[1]} px, 幅 = {ext[2] - ext[0]} px (チャンバー幅80 mm相当)")
plt.tight_layout()
plt.show()

## §4 一次元輝度分布の抽出・保存

チャンバーごとに `lumi1d_{date}_{video}_{chamber}.csv` として `data/luminance_1d/{date}/` に保存する。
CSVの形式は旧コードと同一（行=時刻[5秒間隔]、列=x[pixel]、pandasのデフォルト形式）。
旧ファイル名との対応は下に表示される `legacy_csv` を参照。

In [ ]:
out_dir = OUTPUT_ROOT / cfg["date"]
out_dir.mkdir(parents=True, exist_ok=True)
video_stem = Path(cfg["video_file"]).stem

for ch in CHAMBERS:
    lumi = extract_luminance_from_line(images, cfg["chambers"][ch]["extraction_line_px"])
    out_path = out_dir / f"lumi1d_{cfg['date']}_{video_stem}_{ch}.csv"
    pd.DataFrame(lumi).to_csv(out_path)
    print(f"[{ch}] {np.shape(lumi)} -> {out_path}")
    print(f"       (旧ファイル名: {cfg['chambers'][ch]['legacy_csv']})")

## 補足: 旧コードからの変更点

- **ffmpeg**: `os.chdir(ffmpeg_dir)` 方式を廃止し、PATHにあるffmpegを呼ぶ方式に変更。
- **フレーム順序**: 旧コードは `os.listdir` の返却順（ファイルシステム依存）に頼っていたが、
  ファイル名の秒数による数値ソートに変更し、時系列順を明示的に保証した。
- **フレーム抽出**: 旧コードの「各時刻2フレーム抽出→1枚おき間引き」を、等価な「各時刻1フレーム抽出」に簡略化。
- **座標の外出し**: サンプルごとに異なる較正座標・抽出線・条件・微小回転は
  すべて `config/samples_lowres.json` に集約（旧ノートブック名も `source_notebook` として記録）。
- **CSV命名**: `lumi1d_{date}_{video}_{chamber}.csv` に統一。旧名はconfigの `legacy_csv` を参照。
- **背景除去**: 0320ロットのみ画像レベル背景除去を行う旧手順を`image_level_bg_removal`フラグで再現。
  他ロットは抽出後（03ノートブック側）で一次元分布に対して背景除去を行う。